# Capítulo 3 · Actividad: análisis de datos con NumPy y pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/01_actividad_numpy_pandas.ipynb)

En esta actividad haces de **analista de datos**: vas a responder preguntas sobre datos reales usando solo **NumPy** y **pandas**, sin gráficas. Lo importante no es solo que el código funcione, sino que **sepas interpretar los resultados**.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia. Si editas el original tendrás conflictos al sincronizar tu fork.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```
3. Ten a mano las notebooks de clase: `notebooks/lectures/chapter_03/01_numpy.ipynb` y `02_pandas.ipynb`.

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Puedes añadir todas las celdas que necesites.
- En las celdas ✍️ responde con tus palabras y **apoya tus respuestas con números** de tus resultados.
- Las 💡 **pistas** te recuerdan funciones. Si una función no salió en clase, la pista explica cómo usarla.
- Al terminar, ejecuta **Kernel → Restart & Run All** y comprueba que todo funciona de arriba abajo.

| Parte | Datos | Herramienta |
|---|---|---|
| 1. La cafetería | ventas de un mes (generadas en la notebook) | NumPy |
| 2. Las propinas | `data/tips.csv` | pandas (+ un poco de NumPy) |
| Reto opcional | `data/titanic.csv` | pandas |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

---
## Parte 1 · La cafetería ☕ (NumPy)

Una cafetería vende 5 productos y ha registrado cuántas unidades vendió de cada uno durante **4 semanas** (28 días, de lunes a domingo). La dueña quiere saber cuánto gana, qué días van mejor y qué producto le conviene más.

Ejecuta la siguiente celda para generar los datos. **No hace falta que la modifiques ni que entiendas cada línea.**

In [ ]:
# Datos de la cafetería (no hace falta modificar esta celda)
rng = np.random.default_rng(42)

productos = np.array(["café", "latte", "té", "croissant", "sándwich"])
precios = np.array([1.5, 2.8, 1.2, 2.0, 4.5])       # precio de cada producto, en $

# Unidades vendidas cada día: los fines de semana se vende un 50% más
dia_semana = np.tile(np.arange(7), 4)                # 0 = lunes, ..., 6 = domingo
factor = np.where(dia_semana >= 5, 1.5, 1.0)
promedio = np.outer(factor, [30, 22, 12, 18, 9])     # ventas esperadas por día y producto
ventas_planas = rng.poisson(promedio).ravel()         # ¡todo en una sola fila!

print(ventas_planas[:10], "...")
print("Número de valores:", ventas_planas.size)

### Ejercicio 1 · Dar forma a los datos

`ventas_planas` tiene 140 números seguidos: primero las ventas de los 5 productos del día 1, después las del día 2, y así sucesivamente.

a) Crea una matriz `ventas` con forma `(28, 5)`: una **fila por día** y una **columna por producto** (en el mismo orden que `productos`).

b) Imprime su `shape`, `ndim`, `size` y `dtype`.

c) ¿Cuántos cafés se vendieron el primer día? Comprueba que coincide con `ventas_planas[0]`.

> 💡 **Pista:** `reshape`.

In [ ]:
# Tu código aquí
# a) una fila por día, una columna por producto
ventas = ventas_planas.reshape(28, 5)

# b) características del array
print("Shape:", ventas.shape, "Ndim:", ventas.ndim, "Size:", ventas.size, "Tipo de datos:", ventas.dtype)

# c) cafés del primer día (fila 0, columna 0)
print("Cafés el primer día:", ventas[0, 0], "| coincide con ventas_planas[0]:", ventas[0, 0] == ventas_planas[0])

### Ejercicio 2 · Indexación

Obtén, usando índices y *slicing* sobre `ventas`:

a) Las ventas de todos los productos el **primer día**.

b) Las ventas de **latte** (columna 1) durante todo el mes.

c) Las ventas de la **última semana** (los últimos 7 días). Comprueba que su forma es `(7, 5)`.

d) Las ventas de **croissant y sándwich** (las dos últimas columnas) durante la **primera semana**.

> 💡 **Pista:** `M[filas, columnas]`. Recuerda que los índices negativos cuentan desde el final.

In [ ]:
# Tu código aquí
print("a) Primer día:", ventas[0])
print("b) Latte en todo el mes:", ventas[:, 1])
ultima_semana = ventas[-7:]
print("c) Última semana, forma:", ultima_semana.shape)
print(ultima_semana)
print("d) Croissant y sándwich, primera semana:")
print(ventas[:7, -2:])

### Ejercicio 3 · Ingresos

a) Calcula la matriz `ingresos` (forma `(28, 5)`): lo que se ingresó por cada producto cada día.

b) ¿Cuánto ingresó la cafetería en todo el mes?

c) Crea `ingreso_diario`, un array con el ingreso total de **cada día** (forma `(28,)`).

d) ¿Cuál fue el ingreso diario medio, el máximo y el mínimo?

> 💡 **Pistas:**
> - `ventas * precios` multiplica **cada fila** de `ventas` por el vector `precios`, elemento a elemento: la columna del café por 1.5, la del latte por 2.8… (este mecanismo se llama *broadcasting*).
> - `M.sum()` suma todo. `M.sum(axis=1)` suma **cada fila** y devuelve un resultado por fila; `M.sum(axis=0)` suma **cada columna**.

In [ ]:
# Tu código aquí
# a) broadcasting: cada columna se multiplica por su precio
ingresos = ventas * precios
print("a) Forma de ingresos:", ingresos.shape)

# b) ingreso total del mes
print(f"b) Ingreso del mes: ${ingresos.sum():.2f}")

# c) suma de cada fila (cada día)
ingreso_diario = ingresos.sum(axis=1)
print("c) Forma de ingreso_diario:", ingreso_diario.shape)

# d) media, máximo y mínimo
print(f"d) Media: ${ingreso_diario.mean():.2f} | Máximo: ${ingreso_diario.max():.2f} | Mínimo: ${ingreso_diario.min():.2f}")

### Ejercicio 4 · Días buenos y días flojos

Usa este array con el número de cada día (del 1 al 28):
```python
dias = np.arange(1, 29)
```

a) Crea una máscara con los días cuyo ingreso supera el **ingreso diario medio** y muestra **qué días** fueron.

b) ¿**Cuántos** días fueron?

c) Crea un array que diga `"bueno"` o `"flojo"` para cada día, según si supera o no la media.

d) Calcula `dias[mascara] % 7` (el resto de dividir entre 7). El día 1 fue lunes, así que un resto de **6** es sábado y un resto de **0** es domingo. ¿Qué tienen en común los días buenos?

> 💡 **Pistas:** `.sum()` sobre una máscara cuenta los `True`, porque `True` vale 1 y `False` vale 0. Para c), `np.where`.

In [ ]:
# Tu código aquí
dias = np.arange(1, 29)

# a) días con ingreso por encima de la media
mascara = ingreso_diario > ingreso_diario.mean()
print("a) Días buenos:", dias[mascara])

# b) cuántos
print("b) Número de días buenos:", mascara.sum())

# c) etiqueta para cada día
etiqueta = np.where(mascara, "bueno", "flojo")
print("c)", etiqueta)

# d) resto de dividir entre 7 (6 = sábado, 0 = domingo)
print("d) Restos:", dias[mascara] % 7)

✍️ **Tu respuesta:** 11 de los 28 días superaron la media de $226.52. Sus restos al dividir entre 7 son casi todos 6 (sábado) o 0 (domingo): **los 8 sábados y domingos del mes (días 6, 7, 13, 14, 20, 21, 27 y 28) fueron días buenos**. Los otros 3 días buenos (1, 19 y 23) fueron entre semana. En resumen, lo que tienen en común los días buenos es que son **fin de semana**, cuando se vende un 50% más.

### Ejercicio 5 · Análisis por semanas

a) Reorganiza `ingreso_diario` en una matriz `por_semana` de forma `(4, 7)`: una fila por semana y una columna por día (lunes a domingo).

b) ¿Cuánto se ingresó **cada semana**?

c) ¿Cuál es el ingreso medio de **cada día de la semana**? ¿Qué día se vende más? Usa:
```python
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
```

d) Compara el ingreso medio de un día de **fin de semana** con el de un día **entre semana**. ¿Cuántas veces más se ingresa en fin de semana?

> 💡 **Pistas:** `axis=1` da un resultado por fila (por semana) y `axis=0` uno por columna (por día de la semana). `arr.argmax()` devuelve la **posición** del valor máximo; úsala como índice en `nombres_dias`.

In [ ]:
# Tu código aquí
# a) 4 semanas x 7 días
por_semana = ingreso_diario.reshape(4, 7)
print("a) Forma:", por_semana.shape)

# b) ingreso de cada semana (suma por filas)
print("b) Ingreso por semana:", por_semana.sum(axis=1).round(2))

# c) ingreso medio de cada día de la semana (media por columnas)
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
media_dia = por_semana.mean(axis=0)
for nombre, valor in zip(nombres_dias, media_dia):
    print(f"   {nombre:<10} ${valor:.2f}")
print("   Día que más se vende:", nombres_dias[media_dia.argmax()])

# d) fin de semana (sábado y domingo) frente a entre semana
media_finde = media_dia[5:].mean()
media_semana = media_dia[:5].mean()
print(f"d) Fin de semana: ${media_finde:.2f} | Entre semana: ${media_semana:.2f} | Razón: {media_finde / media_semana:.2f} veces")

### Ejercicio 6 · El producto estrella

a) Calcula el ingreso total de **cada producto** en el mes.

b) ¿Qué **porcentaje** del ingreso total aporta cada producto? Redondea a 1 decimal.

c) ¿Qué producto genera **más dinero**? ¿Y cuál vende **más unidades**?

> 💡 **Pista:** `axis=0` y `argmax`, igual que en el ejercicio anterior.

In [ ]:
# Tu código aquí
# a) ingreso total de cada producto (suma por columnas)
ingreso_producto = ingresos.sum(axis=0)
for nombre, valor in zip(productos, ingreso_producto):
    print(f"a) {nombre:<10} ${valor:.2f}")

# b) porcentaje sobre el total
porcentaje = (ingreso_producto / ingreso_producto.sum() * 100).round(1)
for nombre, valor in zip(productos, porcentaje):
    print(f"b) {nombre:<10} {valor}%")

# c) producto que más dinero deja y producto que más unidades vende
unidades_producto = ventas.sum(axis=0)
print("c) Más dinero:", productos[ingreso_producto.argmax()])
print("   Más unidades:", productos[unidades_producto.argmax()], "->", unidades_producto)

✍️ **Pregunta:** ¿por qué el producto que más se vende no es el que más dinero deja? Si fueras la dueña, ¿qué harías con esta información?

✍️ **Tu respuesta:** El **café** es el producto que más se vende (989 unidades), pero el que más dinero deja es el **latte** ($2018.80, el 31.8% del total, frente a $1483.50 del café, el 23.4%). Esto pasa porque el dinero depende de **unidades × precio**, y el latte cuesta $2.8 frente a $1.5 del café: vende menos unidades (721) pero cada una deja casi el doble. Lo mismo ocurre con el sándwich: es el que menos unidades vende (282) y aun así aporta el 20.0% del ingreso porque cuesta $4.5. Si fuera la dueña, **promocionaría el latte y el sándwich** (por ejemplo, ofreciendo un combo café + croissant que empuje a comprar más productos caros) y **revisaría el té**, que solo aporta el 7.2% del ingreso. Además, reforzaría el personal y el inventario en fin de semana, cuando el ingreso medio sube de $194.93 a $305.49 por día.

---
## Parte 2 · Las propinas 💵 (pandas)

Un camarero anotó, durante varias semanas, cada mesa que atendió: cuánto pagaron, cuánta propina dejaron y algunos datos de la mesa. Queremos responder una pregunta: **¿de qué depende la propina?**

El archivo es `data/tips.csv` y tiene estas columnas:

| Columna | Significado |
|---|---|
| `total_bill` | importe de la cuenta, en $ |
| `tip` | propina, en $ |
| `sex` | sexo de quien pagó (`Female` / `Male`) |
| `smoker` | si en la mesa había fumadores (`Yes` / `No`) |
| `day` | día de la semana (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | `Lunch` (comida) o `Dinner` (cena) |
| `size` | número de personas en la mesa |

### Ejercicio 7 · Primer vistazo

a) Carga el archivo en un DataFrame llamado `propinas`.

b) Explóralo con `shape`, `head`, `info` y `describe`.

c) ¿Cuántas mesas hay de cada día? ¿Y de cada momento (comida/cena)?

In [ ]:
# Tu código aquí
# a) cargar el archivo
propinas = pd.read_csv(DATA_DIR / "tips.csv")

# b) explorar
print("Forma:", propinas.shape)
propinas.head()

In [ ]:
# Tu código aquí
#propinas.info()
#print(propinas.describe())

# c) mesas por día y por momento
#
print(propinas["day"].value_counts())
print(propinas["time"].value_counts())

✍️ **Pregunta:** ¿cuántas mesas hay? ¿Falta algún dato? ¿Cuál fue la cuenta más alta y cuál la mesa más grande?

✍️ **Tu respuesta:** Hay **244 mesas** (7 columnas). No falta ningún dato: `info()` muestra 244 valores no nulos en todas las columnas. La **cuenta más alta** fue de **$50.81** y la **mesa más grande** fue de **6 personas**. Por días hay más mesas el sábado (87) y el domingo (76) que el jueves (62) y sobre todo que el viernes (19); y hay muchas más cenas (176) que comidas (68).

### Ejercicio 8 · Traducir los datos

Para trabajar más cómodos, vamos a pasar todo a español.

a) Renombra las columnas: `total_bill` → `cuenta`, `tip` → `propina`, `sex` → `sexo`, `smoker` → `fumador`, `day` → `dia`, `time` → `momento`, `size` → `personas`.

b) Traduce los valores:
- `dia`: `Thur` → `Jueves`, `Fri` → `Viernes`, `Sat` → `Sábado`, `Sun` → `Domingo`
- `momento`: `Lunch` → `Comida`, `Dinner` → `Cena`
- `sexo`: `Female` → `Mujer`, `Male` → `Hombre`
- `fumador`: `Yes` → `sí`, `No` → `no`

Comprueba el resultado con `head()`.

> 💡 **Pista:** `rename(columns={...})` y `.map({...})`.

In [ ]:
# Tu código aquí
# a) renombrar columnas
propinas = propinas.rename(columns={
    "total_bill": "cuenta", "tip": "propina", "sex": "sexo", "smoker": "fumador",
    "day": "dia", "time": "momento", "size": "personas",
})

# b) traducir valores
propinas["dia"] = propinas["dia"].map({"Thur": "Jueves", "Fri": "Viernes", "Sat": "Sábado", "Sun": "Domingo"})
propinas["momento"] = propinas["momento"].map({"Lunch": "Comida", "Dinner": "Cena"})
propinas["sexo"] = propinas["sexo"].map({"Female": "Mujer", "Male": "Hombre"})
propinas["fumador"] = propinas["fumador"].map({"Yes": "sí", "No": "no"})
propinas.head()

### Ejercicio 9 · Nuevas columnas

Una propina de 5 $ no significa lo mismo en una cuenta de 15 $ que en una de 50 $. Por eso vamos a crear variables más útiles:

a) `pct_propina`: la propina como **porcentaje** de la cuenta.

b) `cuenta_por_persona`: lo que pagó cada persona de la mesa.

c) `tamaño_mesa`: agrupa `personas` en `"1-2"`, `"3-4"` y `"5-6"`.

d) `generosa`: `"sí"` si la propina es del 20% o más, `"no"` en caso contrario.

> 💡 **Pistas:** operaciones entre columnas, `pd.cut` (con `bins=[0, 2, 4, 6]`) y `np.where`.

In [ ]:
# Tu código aquí
# a) propina como % de la cuenta
propinas["pct_propina"] = propinas["propina"] / propinas["cuenta"] * 100

# b) lo que paga cada persona
propinas["cuenta_por_persona"] = propinas["cuenta"] / propinas["personas"]

# c) tamaño de la mesa en tres grupos (los intervalos son (0,2], (2,4], (4,6])
propinas["tamaño_mesa"] = pd.cut(propinas["personas"], bins=[0, 2, 4, 6], labels=["1-2", "3-4", "5-6"])

# d) propina generosa: 20% o más
propinas["generosa"] = np.where(propinas["pct_propina"] >= 20, "sí", "no")
propinas.head()

### Ejercicio 10 · Filtrar y ordenar

a) ¿Cuántas mesas dejaron una propina **generosa** (20% o más)? ¿Qué **proporción** del total representan?

b) ¿Cuántas **cenas de fin de semana** (sábado o domingo) fueron de **4 personas o más**? ¿Cuál fue su porcentaje de propina medio?

c) Muestra las **5 mesas** con mayor `pct_propina`, con las columnas `cuenta`, `propina`, `pct_propina`, `personas` y `dia`.

d) Usa `query` para obtener las mesas con **fumadores** y una cuenta **entre 10 y 20 $**. ¿Cuántas son?

> 💡 **Pistas:** filtros con `&`, `isin` y `sort_values`. Dentro de `query` el texto va entre comillas simples: `"fumador == 'sí'"`.

In [ ]:
# Tu código aquí
# a) propinas generosas
n_generosas = (propinas["generosa"] == "sí").sum()
print(f"a) Generosas: {n_generosas} ({n_generosas / len(propinas):.1%} del total)")

# b) cenas de fin de semana con 4 o más personas
filtro = propinas["dia"].isin(["Sábado", "Domingo"]) & (propinas["momento"] == "Cena") & (propinas["personas"] >= 4)
print(f"b) Mesas: {filtro.sum()} | % de propina medio: {propinas.loc[filtro, 'pct_propina'].mean():.1f}")

# c) las 5 mesas con mayor porcentaje de propina
print("c)")
display(propinas.sort_values("pct_propina", ascending=False)
        [["cuenta", "propina", "pct_propina", "personas", "dia"]].head(5))

# d) fumadores con cuenta entre 10 y 20 $
fumadores = propinas.query("fumador == 'sí' and 10 <= cuenta <= 20")
print("d) Mesas con fumadores y cuenta entre 10 y 20 $:", len(fumadores))

✍️ **Pregunta:** fíjate en la primera mesa del apartado c). ¿Te parece un caso normal? ¿Cómo crees que afecta a la **media** del porcentaje de propina?

✍️ **Tu respuesta:** No, no es un caso normal: es una mesa con una cuenta de solo $7.25 que dejó $5.15 de propina, un **71.0%**, mientras que la mayoría ronda el 16%. Es un **valor atípico** (*outlier*). Como la media tiene en cuenta todos los valores, una sola mesa así la **sube**: en los domingos, la media pasa de 15.97 sin ella a 16.69 con ella. Por eso hay que mirar los datos extremos antes de fiarse de un promedio.

### Ejercicio 11 · Agrupar y resumir

a) Para cada **día**, calcula el número de mesas, la **media** y la **mediana** de `pct_propina`.

b) Para cada **momento** (comida/cena), crea un resumen con columnas con nombre: `n_mesas`, `cuenta_media`, `propina_media` y `pct_medio`.

c) Para cada número de **personas**, calcula el número de mesas y el `pct_propina` medio.

d) Crea una tabla dinámica con el `pct_propina` medio por **sexo** (filas) y **fumador** (columnas).

e) Quédate solo con las mesas del **domingo** y calcula la media y la mediana de `pct_propina` **con** y **sin** la mesa del 71% que encontraste en el ejercicio 10.

> 💡 **Pistas:** `groupby(...).agg([...])`, `agg(nombre=("columna", "función"))` y `pd.pivot_table`. Redondea con `.round(1)` para que se lea mejor.

In [ ]:
# Tu código aquí
# a) por día
propinas.groupby("dia")["pct_propina"].agg(["count", "mean", "median"]).round(1)

In [ ]:
# Tu código aquí
# b) por momento, con columnas con nombre
propinas.groupby("momento").agg(
    n_mesas=("propina", "count"),
    cuenta_media=("cuenta", "mean"),
    propina_media=("propina", "mean"),
    pct_medio=("pct_propina", "mean"),
).round(2)

In [ ]:
# Tu código aquí
# c) por número de personas
propinas.groupby("personas")["pct_propina"].agg(["count", "mean"]).round(1)

In [ ]:
# Tu código aquí
# d) tabla dinámica sexo x fumador
pd.pivot_table(propinas, values="pct_propina", index="sexo", columns="fumador", aggfunc="mean").round(1)

In [ ]:
# Tu código aquí
# e) solo domingos, con y sin la mesa extrema (la del 71%)
domingo = propinas[propinas["dia"] == "Domingo"]
sin_extrema = domingo[domingo["pct_propina"] < domingo["pct_propina"].max()]

resumen = pd.DataFrame({
    "con la mesa extrema": domingo["pct_propina"].agg(["mean", "median"]),
    "sin la mesa extrema": sin_extrema["pct_propina"].agg(["mean", "median"]),
}).round(2)
resumen

✍️ **Preguntas:**
1. En a), ¿en qué días la media es mayor que la mediana? Según e), ¿cuál de las dos medidas se ve más afectada por una mesa extrema? ¿Cuál usarías para describir la propina "típica"?
2. En c), ¿las mesas más grandes dejan más o menos porcentaje de propina? ¿Te fiarías de los resultados para 1, 5 o 6 personas? ¿Por qué?

✍️ **Tu respuesta:**

1. La **media es mayor que la mediana en los cuatro días** (domingo 16.7 vs 16.1, jueves 16.1 vs 15.4, sábado 15.3 vs 15.2 y viernes 17.0 vs 15.6), señal de que unas pocas mesas con propinas muy altas empujan la media hacia arriba. En e) se ve que la **media es la más afectada** por una mesa extrema: cambia 0.72 puntos (16.69 → 15.97), mientras que la mediana casi no se mueve (16.11 → 16.05). Para describir la propina "típica" usaría la **mediana**, porque es más robusta frente a valores extremos.
2. Las mesas más grandes dejan, en general, **menos porcentaje** de propina: 16.6% con 2 personas, 15.2% con 3, 14.6% con 4 y 14.1% con 5. Sin embargo, **no me fiaría de los resultados para 1, 5 o 6 personas**, porque hay muy pocas mesas (4, 5 y 4 respectivamente) y con tan pocos datos una sola mesa cambia mucho la media (por ejemplo, el 21.7% de las mesas de 1 persona sale de solo 4 mesas, una de ellas con un 32.6%). Solo las mesas de 2, 3 y 4 personas (156, 38 y 37 mesas) dan resultados fiables.

### Ejercicio 12 · Cuidado con las comparaciones

Haz una tabla cruzada que cuente cuántas mesas hay de cada **día** (filas) en cada **momento** (columnas).

> 💡 **Pista:** `pd.crosstab`.

In [ ]:
# Tu código aquí
pd.crosstab(propinas["dia"], propinas["momento"])

✍️ **Pregunta:** si comparas las propinas de la comida con las de la cena, ¿estás comparando solo el momento del día, o también otra cosa? ¿Por qué es importante tenerlo en cuenta antes de sacar conclusiones?

✍️ **Tu respuesta:** No, estás comparando también el **día de la semana**. La tabla muestra que casi todas las comidas son de jueves (61 de las 68 comidas; el jueves solo hay 1 cena), mientras que el sábado y el domingo **solo hay cenas** (87 y 76 mesas, ninguna comida). Entonces, si la comida y la cena difieren en propina, no sabríamos si se debe al momento del día o al día de la semana (o al tipo de cliente de cada día). Esto se llama **variable de confusión**, y es importante tenerla en cuenta porque nos puede llevar a conclusiones falsas: un efecto que atribuimos al momento del día podría deberse a otra variable.

### Ejercicio 13 · Combinar tablas

Tenemos otra tabla con información de cada día:
```python
info_dias = pd.DataFrame({
    "dia": ["Jueves", "Viernes", "Sábado", "Domingo"],
    "orden": [4, 5, 6, 7],
    "fin_de_semana": [False, False, True, True],
})
```

a) Añade esta información a `propinas` con `merge`. Comprueba que el número de filas **no cambia**.

b) Compara la **cuenta media** y el **`pct_propina` medio** entre fin de semana y entre semana.

c) Calcula cuánto se facturó en total **cada día**, ordenado cronológicamente (de jueves a domingo).

> 💡 **Pista:** para c), agrupa por `["orden", "dia"]`: `groupby` ordena los grupos por la primera columna.

In [ ]:
# Tu código aquí
info_dias = pd.DataFrame({
    "dia": ["Jueves", "Viernes", "Sábado", "Domingo"],
    "orden": [4, 5, 6, 7],
    "fin_de_semana": [False, False, True, True],
})

# a) merge por la columna "dia" (how="left" conserva todas las mesas)
filas_antes = len(propinas)
propinas = propinas.merge(info_dias, on="dia", how="left")
print(f"a) Filas antes: {filas_antes} | después: {len(propinas)}")

# b) fin de semana vs entre semana
print("b)")
display(propinas.groupby("fin_de_semana")[["cuenta", "pct_propina"]].mean().round(2))

# c) facturación total de cada día, en orden cronológico
print("c)")
display(propinas.groupby(["orden", "dia"])["cuenta"].sum().round(2))

### Ejercicio 14 · De pandas a NumPy

Los modelos de Machine Learning trabajan con arrays de NumPy, así que es habitual pasar de un DataFrame a un array.

a) Crea `X` con las columnas `cuenta`, `propina` y `personas` convertidas a array de NumPy. ¿Qué forma tiene?

b) **Solo con NumPy**, calcula el porcentaje de propina de cada mesa a partir de `X`. Comprueba que su media coincide con la de `propinas["pct_propina"]`.

c) **Solo con NumPy**, ¿cuántas mesas de **2 personas** dejaron **más de 4 $** de propina?

> 💡 **Pistas:** `.to_numpy()`. En `X`, la columna 0 es la cuenta, la 1 la propina y la 2 las personas.

In [ ]:
# Tu código aquí
# a) DataFrame -> array de NumPy
X = propinas[["cuenta", "propina", "personas"]].to_numpy()
print("a) Forma de X:", X.shape)

# b) porcentaje de propina solo con NumPy (propina / cuenta * 100)
pct = X[:, 1] / X[:, 0] * 100
print("b) Media NumPy:", pct.mean().round(4), "| media pandas:", propinas["pct_propina"].mean().round(4),
      "| coinciden:", np.isclose(pct.mean(), propinas["pct_propina"].mean()))

# c) mesas de 2 personas con más de 4 $ de propina
print("c) Mesas:", ((X[:, 2] == 2) & (X[:, 1] > 4)).sum())

### ✍️ Conclusiones de la Parte 2

Imagina que el camarero te pregunta: *"¿qué tengo que hacer para conseguir mejores propinas?"*. Escribe:

1. **Tres hallazgos** sobre las propinas, cada uno apoyado en un número concreto de tus resultados.
2. **Una limitación** de estos datos que haga que debas tener cuidado con tus conclusiones.

✍️ **Tu respuesta:**

**Tres hallazgos:**
1. La propina típica ronda el **16%** de la cuenta (media de 16.08%) y solo **39 de 244 mesas (16.0%)** dejan una propina generosa de 20% o más. Para el camarero, un 20% es un resultado por encima de lo normal.
2. Las mesas **pequeñas** dejan mejor porcentaje que las grandes: 16.6% con 2 personas frente a 14.6% con 4 y 14.1% con 5. Atender bien a las mesas de 2 puede compensar más de lo que parece.
3. El **fin de semana** no mejora el *porcentaje*: la cuenta media es mayor ($20.89 vs $17.56), pero el porcentaje de propina es parecido (15.96% vs 16.33%). Ganará más dinero en fin de semana porque las cuentas son más altas, no porque la gente dé mayor porcentaje. (Además, las mujeres fumadoras dejan más: 18.2% frente a 15.3%-16.1% en los demás grupos).

**Una limitación:** son datos de **un solo camarero y un solo restaurante** (244 mesas), y son **observacionales**: solo vemos asociaciones, no causas. Variables como el día y el momento están mezcladas (el jueves es casi siempre comida y el fin de semana siempre cena), y algunos grupos tienen muy pocas mesas (5-6 personas, viernes) o una mesa extrema (71%), por lo que no deberíamos generalizar ni concluir que cambiar una cosa *causa* mejores propinas.

---
## Reto opcional · El Titanic 🚢

En 1912 el Titanic se hundió en su viaje inaugural. El archivo `data/titanic.csv` tiene información de 891 pasajeros. Las columnas que necesitas:

| Columna | Significado |
|---|---|
| `survived` | 1 si sobrevivió, 0 si no |
| `sex` | `male` / `female` |
| `age` | edad (¡algunas faltan!) |
| `fare` | precio del billete |
| `class` | clase del billete: `First`, `Second`, `Third` |

**Pregunta:** ¿se cumplió el famoso *"las mujeres y los niños primero"*? ¿Importaba la clase del billete?

a) Carga los datos y usa `info()` para ver qué columnas tienen valores faltantes.

b) ¿Qué proporción de pasajeros sobrevivió?

c) Calcula la tasa de supervivencia por **sexo**, por **clase** y en una tabla dinámica **sexo × clase**.

d) Agrupa las edades en `"niño"` (0–12), `"adolescente"` (13–18), `"adulto"` (19–60) y `"mayor"` (más de 60), y calcula la tasa de supervivencia y el número de pasajeros de cada grupo.

e) ¿Cuánto costaba de media el billete en cada clase?

> 💡 **Pistas:**
> - La **media** de una columna de 0 y 1 es la **proporción** de unos: `titanic["survived"].mean()` es la tasa de supervivencia.
> - Para d), `pd.cut` con `bins=[0, 12, 18, 60, 100]`. Los pasajeros sin edad (`NaN`) quedan fuera de los grupos automáticamente.

In [ ]:
# Tu código aquí
# a) cargar y ver valores faltantes
titanic = pd.read_csv(DATA_DIR / "titanic.csv")
titanic.info()

In [ ]:
# Tu código aquí
# b) proporción de supervivientes
print(f"b) Tasa de supervivencia global: {titanic['survived'].mean():.1%}")

# c) por sexo, por clase y sexo x clase
print("\nPor sexo:\n", titanic.groupby("sex")["survived"].mean().round(3))
print("\nPor clase:\n", titanic.groupby("class", observed=True)["survived"].mean().round(3))
print("\nSexo x clase:")
display(pd.pivot_table(titanic, values="survived", index="sex", columns="class", aggfunc="mean", observed=True).round(3))

In [ ]:
# Tu código aquí
# d) grupos de edad (los pasajeros sin edad quedan fuera)
titanic["grupo_edad"] = pd.cut(titanic["age"], bins=[0, 12, 18, 60, 100],
                               labels=["niño", "adolescente", "adulto", "mayor"])
print("d)")
display(titanic.groupby("grupo_edad", observed=True)["survived"].agg(tasa="mean", n_pasajeros="count").round(3))

# e) precio medio del billete por clase
print("e)")
display(titanic.groupby("class", observed=True)["fare"].mean().round(2))

✍️ **Pregunta:** ¿se cumplió *"las mujeres y los niños primero"*? ¿Y la clase, importaba? Responde con números.

✍️ **Tu respuesta:** **Sí, se cumplió lo de "las mujeres primero"**: sobrevivió el **74.2% de las mujeres** frente a solo el **18.9% de los hombres**. En cuanto a los niños, los de 12 años o menos tuvieron la mayor tasa de supervivencia (**58.0%**), frente a un 38.9% de los adultos y un 22.7% de los mayores, pero hay que tomarlo con cuidado: es solo una muestra de 69 niños, y las edades faltantes (177 pasajeros sin edad) no están incluidas. **La clase también importaba mucho**: sobrevivió el 63.0% de primera clase, el 47.3% de segunda y solo el 24.2% de tercera. Al cruzar sexo y clase se ve que ambas variables influyeron: el 96.8% de las mujeres de primera clase sobrevivió, frente al 50.0% de las de tercera, y solo el 13.5% de los hombres de tercera. El billete de primera costaba de media $84.15, frente a $13.68 en tercera.